In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill


SOURCE_1 = Path("seea_account_availability_assessment.xlsx")
SOURCE_2 = Path("availability_SEEAKB.xlsx")
WORLD_BANK_CLASSIFICATION = Path("worldbank_classification.csv")

SHEET_NAME = "availability_matrix"
COUNTRY_COLUMN = "Country or area"
REGION_COLUMN = "Region"

OUTPUT_XLSX = Path("combined_availability_seea.xlsx")
OUTPUT_CSV = Path("combined_availability_seea.csv")


def read_matrix(path):
    df = pd.read_excel(path, sheet_name=SHEET_NAME)
    if COUNTRY_COLUMN not in df.columns:
        raise ValueError(f"{COUNTRY_COLUMN!r} not found in {path}")
    df[COUNTRY_COLUMN] = df[COUNTRY_COLUMN].astype(str).str.strip()
    return df


def clean_binary(value):
    if pd.isna(value):
        return 0
    if isinstance(value, str):
        value = value.strip().lower()
        if value in {"x", "yes", "true"}:
            return 1
        if value in {"", "no", "false"}:
            return 0
    try:
        return 1 if float(value) > 0 else 0
    except Exception:
        return 0


def normalize_country_name(value):
    value = str(value or "").strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


COUNTRY_ALIASES = {
    "congo": "congo rep",
    "czechia": "czech republic",
    "lao people s democratic republic": "lao pdr",
    "republic of korea": "south korea",
    "republic of moldova": "moldova",
    "russian federation": "russia",
    "slovakia": "slovak republic",
    "state of palestine": "west bank and gaza",
    "turkiye": "turkey",
    "united kingdom of great britain and northern ireland": "united kingdom",
    "united republic of tanzania": "tanzania",
    "united states of america": "united states",
}


DIRECT_REGION_ALIASES = {
    "global": "Global",
    "montserrat": "Latin America & Caribbean",
}


def load_region_lookup(path):
    if not path.exists():
        raise FileNotFoundError(f"World Bank classification file not found: {path.resolve()}")

    classification = pd.read_csv(path)
    required = {"Country", REGION_COLUMN}
    missing = required - set(classification.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")

    lookup = {
        normalize_country_name(row["Country"]): row[REGION_COLUMN]
        for _, row in classification.iterrows()
        if pd.notna(row["Country"])
    }
    return classification, lookup


def add_region_column(combined, region_lookup):
    regions = []
    unmatched_rows = []

    for country in combined[COUNTRY_COLUMN]:
        normalized = normalize_country_name(country)
        lookup_key = COUNTRY_ALIASES.get(normalized, normalized)

        if normalized in DIRECT_REGION_ALIASES:
            region = DIRECT_REGION_ALIASES[normalized]
            match_rule = "direct_region_alias"
        else:
            region = region_lookup.get(lookup_key)
            match_rule = "exact" if lookup_key == normalized else "country_alias"

        if pd.isna(region) or region is None:
            region = ""
            unmatched_rows.append(
                {
                    "country": country,
                    "normalized_country": normalized,
                    "lookup_key": lookup_key,
                    "match_rule": "unmatched",
                }
            )

        regions.append(region)

    output = combined.copy()
    if REGION_COLUMN in output.columns:
        output = output.drop(columns=[REGION_COLUMN])

    insert_at = output.columns.get_loc(COUNTRY_COLUMN) + 1
    output.insert(insert_at, REGION_COLUMN, regions)
    unmatched = pd.DataFrame(unmatched_rows)
    return output, unmatched


def combine_matrices(df1, df2):
    all_columns = list(df1.columns)
    for column in df2.columns:
        if column not in all_columns:
            all_columns.append(column)

    df1 = df1.reindex(columns=all_columns)
    df2 = df2.reindex(columns=all_columns)

    countries = sorted(set(df1[COUNTRY_COLUMN]) | set(df2[COUNTRY_COLUMN]))
    one = df1.set_index(COUNTRY_COLUMN)
    two = df2.set_index(COUNTRY_COLUMN)

    combined_rows = []
    difference_rows = []

    for country in countries:
        row = {COUNTRY_COLUMN: country}

        for column in all_columns:
            if column == COUNTRY_COLUMN:
                continue

            v1 = one[column].get(country, 0) if column in one.columns else 0
            v2 = two[column].get(country, 0) if column in two.columns else 0

            if column.endswith("_reported"):
                b1 = clean_binary(v1)
                b2 = clean_binary(v2)

                # Keep reported values unchanged from the first file when present.
                # For countries only present in the second file, use the second file.
                if country in one.index:
                    row[column] = b1
                else:
                    row[column] = b2

                if country in one.index and country in two.index and b1 != b2:
                    difference_rows.append(
                        {
                            "country": country,
                            "column": column,
                            "source_1_value": b1,
                            "source_2_value": b2,
                            "combined_value": row[column],
                            "rule": "reported kept from source_1",
                        }
                    )
            else:
                row[column] = min(clean_binary(v1) + clean_binary(v2), 1)

        combined_rows.append(row)

    combined = pd.DataFrame(combined_rows, columns=all_columns)
    differences = pd.DataFrame(difference_rows)
    return combined, differences


def write_outputs(combined, differences, unmatched_regions):
    combined.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    summary = pd.DataFrame(
        [
            {"item": "source_1", "value": str(SOURCE_1)},
            {"item": "source_2", "value": str(SOURCE_2)},
            {"item": "countries_combined", "value": len(combined)},
            {"item": "columns_combined", "value": len(combined.columns)},
            {"item": "world_bank_classification", "value": str(WORLD_BANK_CLASSIFICATION)},
            {"item": "region_rule", "value": "Matched Country or area to worldbank_classification Country and inserted Region after Country or area."},
            {"item": "reported_rule", "value": "Kept from source_1 when country exists there; otherwise source_2."},
            {"item": "available_nonreport_rule", "value": "Combined as min(source_1 + source_2, 1)."},
            {"item": "reported_differences_logged", "value": len(differences)},
            {"item": "unmatched_regions", "value": len(unmatched_regions)},
        ]
    )

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        combined.to_excel(writer, sheet_name="availability_matrix", index=False)
        summary.to_excel(writer, sheet_name="combination_summary", index=False)
        differences.to_excel(writer, sheet_name="reported_differences", index=False)
        unmatched_regions.to_excel(writer, sheet_name="unmatched_regions", index=False)

        for worksheet in writer.book.worksheets:
            worksheet.freeze_panes = "A2"
            header_fill = PatternFill("solid", fgColor="0F766E")
            header_font = Font(color="FFFFFF", bold=True)
            for cell in worksheet[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

            for column_cells in worksheet.columns:
                max_len = max(len(str(cell.value or "")) for cell in column_cells[:2000])
                worksheet.column_dimensions[column_cells[0].column_letter].width = min(max(max_len + 2, 12), 55)


def main():
    df1 = read_matrix(SOURCE_1)
    df2 = read_matrix(SOURCE_2)
    _, region_lookup = load_region_lookup(WORLD_BANK_CLASSIFICATION)
    combined, differences = combine_matrices(df1, df2)
    combined, unmatched_regions = add_region_column(combined, region_lookup)
    write_outputs(combined, differences, unmatched_regions)

    available_cols = [c for c in combined.columns if c.endswith("_available")]
    nonreport_cols = [c for c in combined.columns if c.endswith("_nonreport_avail")]

    print("Done.")
    print(f"Source 1 rows: {len(df1)}")
    print(f"Source 2 rows: {len(df2)}")
    print(f"Combined countries: {len(combined)}")
    print(f"Combined columns: {len(combined.columns)}")
    print(f"Total available flags: {int(combined[available_cols].sum().sum())}")
    print(f"Total nonreported available flags: {int(combined[nonreport_cols].sum().sum())}")
    print(f"Reported differences logged: {len(differences)}")
    print(f"Unmatched regions: {len(unmatched_regions)}")
    print(f"Excel output: {OUTPUT_XLSX.resolve()}")
    print(f"CSV output: {OUTPUT_CSV.resolve()}")


if __name__ == "__main__":
    main()
